In [ ]:
from pathlib import Path
import re
import shutil
import traceback

import numpy as np
import pandas as pd

from pyspoc import Calculator, Config

octave not found, please see README
Octave not available: octave not found, please see README
Starting JVM with java class /Users/ilg/Desktop/year4/M4R/py-spoc/pyspoc/lib/jidt/infodynamics.jar.


In [ ]:
# ============================================================
# Paths
# ============================================================

PROJECT_DIR = Path("/Users/ilg/Desktop/year4/M4R/python_files")

REAL_WORLD_DIR = PROJECT_DIR / "real_world_datasets"
DATA_DIR = REAL_WORLD_DIR / "data"

SYNTHETIC_DIR = REAL_WORLD_DIR

METADATA_PATH = REAL_WORLD_DIR / "real_world_dataset_metadata.csv"

CONFIG_PATH = PROJECT_DIR / "qi_config.yaml"

OUTPUT_DIR = PROJECT_DIR / "meta_raw_unnormalised"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "real_world_raw_statistics.csv"
CHECKPOINT_CSV = OUTPUT_DIR / "real_world_raw_statistics_checkpoint.csv"
ERROR_CSV = OUTPUT_DIR / "real_world_raw_statistics_errors.csv"


# ============================================================
# Settings
# ============================================================

EXPECTED_N = 1000
EXPECTED_P = 100

DATA_EXTENSIONS = {".csv", ".npy", ".npz"}

ALLOW_CROP_TO_EXPECTED_SIZE = False

In [ ]:
# ============================================================
# Helper functions
# ============================================================

def safe_id_from_path(path: Path) -> str:
    """
    Make a traceable dataset ID from the relative file path.
    This avoids relying on known dataset folder names.
    """
    rel = path.relative_to(DATA_DIR)
    text = str(rel.with_suffix(""))
    text = re.sub(r"[^\w\-.]+", "__", text)
    text = re.sub(r"__+", "__", text)
    return text.strip("_")


def is_data_file(path: Path) -> bool:
    """
    Keep only actual matrix files.
    Ignore metadata, reports, checkpoints and index npz files.
    """
    if not path.is_file():
        return False

    low = path.name.lower()

    if path.suffix.lower() not in DATA_EXTENSIONS:
        return False

    if low.endswith("__indices.npz"):
        return False

    bad_keywords = [
        "metadata",
        "manifest",
        "report",
        "checkpoint",
        "error",
        "readme",
        "license",
    ]
    if any(k in low for k in bad_keywords):
        return False

    return True


def paired_indices_path(data_path: Path) -> str:
    """
    For a data block like:
        xxx.csv
    look for:
        xxx__indices.npz
    """
    possible = data_path.with_name(data_path.stem + "__indices.npz")
    if possible.exists():
        return str(possible)
    return ""


def discover_real_world_datasets() -> pd.DataFrame:
    """
    Recursively discover all subsampled real-world dataset files.
    No dataset-specific folder names are assumed.
    """
    records = []

    if not DATA_DIR.exists():
        raise FileNotFoundError(f"DATA_DIR does not exist: {DATA_DIR}")

    for path in sorted(DATA_DIR.rglob("*")):
        if not is_data_file(path):
            continue

        dataset_id = safe_id_from_path(path)

        records.append({
            "dataset_id": dataset_id,
            "file_path": str(path),
            "relative_path": str(path.relative_to(DATA_DIR)),
            "indices_npz_path": paired_indices_path(path),
            "file_type": path.suffix.lower(),
        })

    metadata = pd.DataFrame(records)

    if metadata.empty:
        raise RuntimeError(
            f"No usable dataset files found under {DATA_DIR}. "
            "Expected .csv, .npy, or .npz files."
        )

    metadata.to_csv(METADATA_PATH, index=False)
    print(f"Discovered {len(metadata)} real-world dataset files.")
    print(f"Metadata saved to: {METADATA_PATH}")

    return metadata


def load_csv_matrix(path: Path) -> np.ndarray:
    """
    Load a CSV matrix robustly.
    Drops accidental index columns such as 'Unnamed: 0'.
    Requires numeric entries after loading.
    """
    df = pd.read_csv(path)

    # Drop accidental saved index columns.
    unnamed_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    # Convert all columns to numeric.
    numeric_df = df.apply(pd.to_numeric, errors="coerce")

    # Drop columns that became entirely NaN.
    numeric_df = numeric_df.dropna(axis=1, how="all")

    # If any remaining NaNs exist, impute by column median.
    if numeric_df.isna().any().any():
        numeric_df = numeric_df.fillna(numeric_df.median(numeric_only=True))

    X = numeric_df.to_numpy(dtype=float)

    return X


def load_npz_matrix(path: Path) -> np.ndarray:
    """
    Load a data matrix from npz.
    Ignores index npz files by design before this function is called.
    Uses the first 2D numeric array inside the archive.
    """
    archive = np.load(path, allow_pickle=True)

    for key in archive.files:
        arr = archive[key]
        if isinstance(arr, np.ndarray) and arr.ndim == 2 and np.issubdtype(arr.dtype, np.number):
            return arr.astype(float)

    raise ValueError(f"No 2D numeric array found inside npz file: {path}")


def load_dataset_matrix(file_path: str) -> np.ndarray:
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        X = load_csv_matrix(path)
    elif suffix == ".npy":
        X = np.load(path).astype(float)
    elif suffix == ".npz":
        X = load_npz_matrix(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

    if X.ndim != 2:
        raise ValueError(f"Dataset is not 2D. Shape found: {X.shape}")

    if not np.all(np.isfinite(X)):
        raise ValueError("Dataset contains NaN, +inf or -inf after loading.")

    if X.shape != (EXPECTED_N, EXPECTED_P):
        if ALLOW_CROP_TO_EXPECTED_SIZE and X.shape[0] >= EXPECTED_N and X.shape[1] >= EXPECTED_P:
            X = X[:EXPECTED_N, :EXPECTED_P]
        else:
            raise ValueError(
                f"Expected shape {(EXPECTED_N, EXPECTED_P)}, but found {X.shape}."
            )

    return X


def load_pyspoc_config():
    """
    Compatible with PySPoC versions using either:
        Config.from_yaml_file(name, path)
    or:
        Config.from_yaml_file(path)
    """
    try:
        return Config.from_yaml_file("qi_config", str(CONFIG_PATH))
    except TypeError:
        return Config.from_yaml_file(str(CONFIG_PATH))


def flatten_or_preserve_results(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep PySPoC's two-level column structure if it exists.
    If columns are flat, convert them to a two-level structure.
    """
    result_df = result_df.reset_index(drop=True)

    if isinstance(result_df.columns, pd.MultiIndex):
        return result_df

    result_df.columns = pd.MultiIndex.from_tuples(
        [("statistic", str(c)) for c in result_df.columns]
    )
    return result_df


def make_metadata_row(row: pd.Series, X: np.ndarray) -> pd.DataFrame:
    meta = {
        "dataset_id": row["dataset_id"],
        "file_path": row["file_path"],
        "relative_path": row["relative_path"],
        "indices_npz_path": row.get("indices_npz_path", ""),
        "file_type": row.get("file_type", ""),
        "n": X.shape[0],
        "p": X.shape[1],
    }

    meta_df = pd.DataFrame([meta])
    meta_df.columns = pd.MultiIndex.from_tuples(
        [("metadata", c) for c in meta_df.columns]
    )
    return meta_df


def read_checkpoint_if_exists() -> pd.DataFrame | None:
    if not CHECKPOINT_CSV.exists():
        return None

    try:
        df = pd.read_csv(CHECKPOINT_CSV, header=[0, 1])
        print(f"Loaded checkpoint with {len(df)} completed rows.")
        return df
    except Exception:
        print("Checkpoint exists but could not be read with two-row header. Ignoring checkpoint.")
        return None


def completed_dataset_ids(checkpoint_df: pd.DataFrame | None) -> set:
    if checkpoint_df is None or checkpoint_df.empty:
        return set()

    try:
        return set(checkpoint_df[("metadata", "dataset_id")].astype(str))
    except Exception:
        return set()

In [ ]:
# ============================================================
# Main statistics pipeline
# ============================================================

def run_real_world_statistics_pipeline():
    metadata = discover_real_world_datasets()
    cfg = load_pyspoc_config()

    checkpoint_df = read_checkpoint_if_exists()
    done_ids = completed_dataset_ids(checkpoint_df)

    error_rows = []

    for idx, row in metadata.iterrows():
        dataset_id = str(row["dataset_id"])

        if dataset_id in done_ids:
            print(f"[{idx + 1}/{len(metadata)}] Skipping completed: {dataset_id}")
            continue

        print(f"\n[{idx + 1}/{len(metadata)}] Processing {dataset_id}")

        try:
            X = load_dataset_matrix(row["file_path"])

            calc = Calculator(X)
            calc.compute(cfg)

            stats_df = flatten_or_preserve_results(calc.results)
            meta_df = make_metadata_row(row, X)

            out_row = pd.concat([meta_df, stats_df], axis=1)

            if checkpoint_df is None:
                checkpoint_df = out_row
            else:
                checkpoint_df = pd.concat([checkpoint_df, out_row], ignore_index=True)

            checkpoint_df.to_csv(CHECKPOINT_CSV, index=False)
            print(f"Saved checkpoint: {CHECKPOINT_CSV}")

        except Exception as e:
            print(f"FAILED: {dataset_id}")
            print(e)

            error_rows.append({
                "dataset_id": dataset_id,
                "file_path": row["file_path"],
                "relative_path": row["relative_path"],
                "reason": str(e),
                "traceback": traceback.format_exc(),
            })

            pd.DataFrame(error_rows).to_csv(ERROR_CSV, index=False)

    if checkpoint_df is not None:
        checkpoint_df.to_csv(OUTPUT_CSV, index=False)
        shutil.copyfile(OUTPUT_CSV, CHECKPOINT_CSV)

    print("\nDone.")
    print(f"Output CSV: {OUTPUT_CSV}")
    print(f"Checkpoint CSV: {CHECKPOINT_CSV}")
    print(f"Metadata CSV: {METADATA_PATH}")
    print(f"Error CSV: {ERROR_CSV}")


run_real_world_statistics_pipeline()

Discovered 256 real-world dataset files.
Metadata saved to: /Users/ilg/Desktop/year4/M4R/python_files/real_world_datasets/real_world_dataset_metadata.csv
Registering YAML configuration file: /Users/ilg/Desktop/year4/M4R/python_files/qi_config.yaml.
Building internal configuration.
  ✔ Module pyspoc.statistics.basic loaded successfully.
    ✔ Statistic Covariance scheme 'feature_covariance_matrix' added successfully.
    ✔ Statistic Precision scheme 'feature_precision_matrix' added successfully.
    ✔ Statistic KendallTau scheme 'feature_kendall_tau_signed' added successfully.
    ✔ Statistic SpearmanR scheme 'feature_spearman_r_signed' added successfully.
/Users/ilg/Desktop/year4/M4R/python_files/Plaid_metrics/Plaid_metrics.py
    ✔ ReducedStatistic PlaidAllMetrics scheme 'assumed_k2' added successfully.
    ✔ ReducedStatistic PlaidAllMetrics scheme 'assumed_k3' added successfully.
    ✔ ReducedStatistic PlaidAllMetrics scheme 'assumed_k5' added successfully.
/Users/ilg/Desktop/year4/M

Error importing in API mode: ImportError("dlopen(/Users/ilg/anaconda3/envs/pyspoc_env/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <85D47EFD-9B5F-368A-BC00-EBCFE4080F31> /Users/ilg/anaconda3/envs/pyspoc_env/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


    ✔ ReducedStatistic FABIAAllMetrics scheme 'assumed_k2' added successfully.
    ✔ ReducedStatistic FABIAAllMetrics scheme 'assumed_k3' added successfully.
    ✔ ReducedStatistic FABIAAllMetrics scheme 'assumed_k5' added successfully.
/Users/ilg/Desktop/year4/M4R/python_files/summary_stats/distance_distrib_stats.py
    ✔ ReducedStatistic DistanceDistributionBasicSummarize scheme 'cosine_distrib_meancentroid' added successfully.
    ✔ ReducedStatistic DistanceDistributionBasicSummarize scheme 'euclidean_distrib_meancentroid' added successfully.
    ✔ ReducedStatistic DistanceDistributionBasicSummarize scheme 'canberra_distrib_meancentroid' added successfully.
    ✔ ReducedStatistic DistanceDistributionBasicSummarize scheme 'cosine_skew_meancentroid' added successfully.
    ✔ ReducedStatistic DistanceDistributionBasicSummarize scheme 'euclidean_skew_meancentroid' added successfully.
    ✔ ReducedStatistic DistanceDistributionBasicSummarize scheme 'canberra_kurt_meancentroid' added succ

Processing [None: pyspoc.statistics.basic.Precision.feature_precision_matrix]:  25%|██▌       | 1/4 [00:00<00:00,  6.49it/s]  /Users/ilg/Desktop/year4/M4R/py-spoc/pyspoc/calculator.py:167: UserWarning: Caught <class 'AttributeError'> for Statistic "pyspoc.statistics.basic.Precision.feature_precision_matrix": 'Precision' object has no attribute '_Precision__fit'
  warnings.warn(f'Caught {type(e)} for Statistic "{stat_name}": {e}')
Processing [None: pyspoc.statistics.basic.SpearmanR.feature_spearman_r_signed]:  75%|███████▌  | 3/4 [00:07<00:02,  2.80s/it]  /Users/ilg/Desktop/year4/M4R/py-spoc/pyspoc/statistics/basic.py:148: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr = sp.stats.spearmanr(x, y).correlation
Processing [None: pyspoc.statistics.basic.SpearmanR.feature_spearman_r_signed]: 100%|██████████| 4/4 [00:12<00:00,  3.00s/it]
Processing [None: pyspoc.reducers.basic.Determinant.scaled_matrix_determinant]:   0%|          | 0/12 [0

Cycle: 500


...

Processing [None: /Users/ilg/Desktop/year4/M4R/python_files/summary_stats/distance_distrib_stats.py.DistanceDistributionBasicSummarize.cosine_skew_pairwise]:  55%|█████▍    | 17/31 [00:02<00:01, 11.08it/s]     

Processing [None: /Users/ilg/Desktop/year4/M4R/python_files/summary_stats/distance_distrib_stats.py.DistanceDistributionBasicSummarize.canberra_kurt_pairwise]:  55%|█████▍    | 17/31 [00:02<00:01, 11.08it/s] 

Processing [None: /Users/ilg/Desktop/year4/M4R/python_files/summary_stats/distance_distrib_stats.py.DistanceDistributionModesSummarize.corr_peaks_meancentroid]:  65%|██████▍   | 20/31 [00:02<00:00, 12.62it/s]

Processing [None: pyspoc.rstatistics.pca.PCAVarianceExplainedRatio.pca_variance_ratio_top_1_2_3]: 100%|██████████| 31/31 [00:09<00:00,  3.26it/s]                                                                    



Calculation complete. Time taken: 14.1275s
Saved checkpoint: /Users/ilg/Desktop/year4/M4R/python_files/meta_raw_unnormalised/real_world_raw_statistics_checkpoint.csv

Done.
Output CSV: /Users/ilg/Desktop/year4/M4R/python_files/meta_raw_unnormalised/real_world_raw_statistics.csv
Checkpoint CSV: /Users/ilg/Desktop/year4/M4R/python_files/meta_raw_unnormalised/real_world_raw_statistics_checkpoint.csv
Metadata CSV: /Users/ilg/Desktop/year4/M4R/python_files/real_world_datasets/real_world_dataset_metadata.csv
Error CSV: /Users/ilg/Desktop/year4/M4R/python_files/meta_raw_unnormalised/real_world_raw_statistics_errors.csv
